### 解压数据集

In [1]:
! unzip image_dataset.zip

Archive:  image_dataset.zip
   creating: image_dataset/
  inflating: image_dataset/zx_-0.000000_-0.000000_763439c8-0dd6-11f0-ae2f-50290c18d70c.jpg  
  inflating: image_dataset/zx_-0.000000_-0.000000_76489a44-0dd6-11f0-ae2f-50290c18d70c.jpg  
  inflating: image_dataset/zx_-0.000000_-0.000000_765d09ca-0dd6-11f0-ae2f-50290c18d70c.jpg  
  inflating: image_dataset/zx_-0.000000_-0.000000_767180da-0dd6-11f0-ae2f-50290c18d70c.jpg  
  inflating: image_dataset/zx_-0.000000_-0.000000_76857efa-0dd6-11f0-ae2f-50290c18d70c.jpg  
  inflating: image_dataset/zx_-0.000000_-0.000000_7699b5dc-0dd6-11f0-ae2f-50290c18d70c.jpg  
  inflating: image_dataset/zx_-0.000000_-0.000000_76ae10d6-0dd6-11f0-ae2f-50290c18d70c.jpg  
  inflating: image_dataset/zx_-0.000000_-0.000000_76c25ac8-0dd6-11f0-ae2f-50290c18d70c.jpg  
  inflating: image_dataset/zx_-0.000000_-0.000000_76d69ea2-0dd6-11f0-ae2f-50290c18d70c.jpg  
  inflating: image_dataset/zx_-0.000000_-0.000000_76eae650-0dd6-11f0-ae2f-50290c18d70c.jpg  
  inflating: i

### 开始训练

In [2]:
import torch
import torch.optim as optim
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
import glob
import PIL.Image
import os
import numpy as np
dir = './'
data = os.listdir(dir+'image_dataset/')
# 创建一个torch.utils.data.Dataset的实现。因为模型输入为224*224，图像分辨率为960*224所以X方向坐标需要缩放
def get_x(path):
    """Gets the x value from the image filename"""
    return (float(path.split("_")[1]) / 1.5) 

def get_y(path):
    """Gets the y value from the image filename"""
    return (float(path.split("_")[2])/0.2)

class XYDataset(torch.utils.data.Dataset):

    def __init__(self, directory, random_hflips=False):
        self.directory = directory
        self.random_hflips = random_hflips
        self.image_paths = glob.glob(os.path.join(self.directory, '*.jpg'))
        self.color_jitter = transforms.ColorJitter(0.3, 0.3, 0.3, 0.3)
        #color_jitter 数据增强，brightness, contrast,sharpness, color
        
    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
 
        image = PIL.Image.open(image_path)
        x = float(get_x(os.path.basename(image_path)))
        y = float(get_y(os.path.basename(image_path)))

        if self.random_hflips:
          if float(np.random.rand(1)) > 0.5:
              image = transforms.functional.hflip(image)
              x = -x

        image = self.color_jitter(image)
        image = transforms.functional.resize(image, (224, 224))
        image = transforms.functional.to_tensor(image)
        image = image.numpy().copy()
        image = torch.from_numpy(image)
        image = transforms.functional.normalize(image,
                [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        return image, torch.tensor([x, y]).float()

def main(args=None):
    # 需要根据自己的环境改为数据集存放位置
    dataset = XYDataset(dir+'image_dataset', random_hflips=False)

    print(len(dataset))
    # 创建训练集和测试集
    test_percent = 0.1
    num_test = int(test_percent * len(dataset))
    train_dataset, test_dataset = torch.utils.data.random_split(dataset, [len(dataset) - num_test, num_test])

    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=24,
        shuffle=True,
        num_workers=0
    )

    test_loader = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=24,
        shuffle=True,
        num_workers=0
    )

    # 创建ResNet18模型，这里选用已经预训练的模型，
    # 更改fc输出为2，即x、y坐标值
    model = models.resnet18(weights=True)
    model.fc = torch.nn.Linear(512, 2)
    device = torch.device("cuda:0" if torch.cuda.is_available() else 'cpu')
   
    model = model.to(device)

    NUM_EPOCHS = 100
    BEST_MODEL_PATH = dir+'best_line_follower_model_xy.pth'
    best_loss = 1e9

    optimizer = optim.Adam(model.parameters())

    for epoch in range(NUM_EPOCHS):

        model.train()
        train_loss = 0.0
        for images, labels in iter(train_loader):
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = F.mse_loss(outputs, labels)
            train_loss += float(loss)
            loss.backward()
            optimizer.step()
        train_loss /= len(train_loader)

        model.eval()
        test_loss = 0.0
        for images, labels in iter(test_loader):
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = F.mse_loss(outputs, labels)
            test_loss += float(loss)
        test_loss /= len(test_loader)

        print('Epoch:%d,Train loss:%f,Val loss:%f' % (epoch, train_loss, test_loss))
        if test_loss < best_loss:
            print("save")
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            best_loss = test_loss

if __name__ == '__main__':
    main()

1905
Epoch:0,Train loss:0.186756,Val loss:0.101412
save
Epoch:1,Train loss:0.090508,Val loss:0.072073
save
Epoch:2,Train loss:0.076326,Val loss:0.075477
Epoch:3,Train loss:0.068205,Val loss:0.054558
save
Epoch:4,Train loss:0.052989,Val loss:0.058543
Epoch:5,Train loss:0.051830,Val loss:0.085415
Epoch:6,Train loss:0.044361,Val loss:0.057137
Epoch:7,Train loss:0.039679,Val loss:0.042621
save
Epoch:8,Train loss:0.035603,Val loss:0.051988
Epoch:9,Train loss:0.035850,Val loss:0.052116
Epoch:10,Train loss:0.028237,Val loss:0.045355
Epoch:11,Train loss:0.028943,Val loss:0.046157
Epoch:12,Train loss:0.027600,Val loss:0.044949
Epoch:13,Train loss:0.022206,Val loss:0.046001
Epoch:14,Train loss:0.022674,Val loss:0.041029
save
Epoch:15,Train loss:0.016625,Val loss:0.040080
save
Epoch:16,Train loss:0.015480,Val loss:0.037773
save
Epoch:17,Train loss:0.014792,Val loss:0.044283
Epoch:18,Train loss:0.016123,Val loss:0.046424
Epoch:19,Train loss:0.012150,Val loss:0.038927
Epoch:20,Train loss:0.015949,V

### 转成onnx模型

In [7]:
!pip install onnx==1.18.0 -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 109.1 MB/s eta 0:00:0000:0100:01


In [3]:
import torchvision
import torch
import onnx
dir = './'
model = torchvision.models.resnet18(weights=False)
model.fc = torch.nn.Linear(512,2)
model.load_state_dict(torch.load(dir+'best_line_follower_model_xy.pth'))
device = torch.device("cuda:0" if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()
x = torch.randn(1, 3, 224, 224, requires_grad=True).to(device)
torch_out = model(x)
torch.onnx.export(model,
                    x,
                    dir+'best_line_follower_model_xy.onnx',
                    export_params=True,
                    opset_version=11,
                    do_constant_folding=True,
                    input_names=['input'],
                    output_names=['output'])
net = onnx.load(dir+'best_line_follower_model_xy.onnx')
onnx.checker.check_model(net)
#onnx.helper.printable_graph(net.graph)

C:\Users\hhch\anaconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
